## GNN Lesson 1: Representing Proteins as Graphs
### What you'll learn
- The fundamental data structure of a graph: nodes, edges, node features.
- How to represent a protein as a PyTorch Geometric `Data` object.
- Two common graph constructions for proteins:
    1. Sequence graph  -- edges between residues adjacent in the sequence.
    2. Contact graph   -- edges between residues close in 3D space.

### Why proteins are graphs
A protein is a chain of amino acids that folds into a 3D structure. Two
residues that are FAR APART in the sequence can be CLOSE in 3D — and that
spatial proximity often determines function (active sites, binding pockets,
etc.). A graph captures this naturally:

    Nodes         = residues
    Node features = amino acid identity (one-hot), or pLM embeddings
    Edges         = sequential or spatial relationships
    Edge features = sequence distance, 3D distance, etc.

### PyTorch Geometric's Data object
Every graph in PyG is a `Data` object with these tensors:

    x          : (num_nodes, num_features)   -- node features
    edge_index : (2, num_edges)              -- pairs (source, target)
    edge_attr  : (num_edges, num_features)   -- optional edge features
    y          : labels (graph- or node-level)
    pos        : (num_nodes, 3)              -- 3D coordinates (optional)

`edge_index` is the unusual bit: it's a (2, num_edges) tensor, not a square
adjacency matrix. The first row holds source-node indices, the second row
holds target-node indices. This is much more memory-efficient for sparse graphs.

> **Run order matters.** The cells below build on each other. Run them **top to bottom** (Jupyter: *Run → Run All Cells*; VS Code: *Run All*). If you hit `NameError: name 'torch' is not defined` (or similar), you skipped the **Setup** cell — run it first.

## Setup — imports & configuration

**Run this cell first.** It imports every library and defines the module-level constants the rest of the notebook relies on.

In [ ]:
import os
import numpy as np
import torch
from torch_geometric.data import Data
import networkx as nx
import matplotlib.pyplot as plt
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_IDX = {a: i for i, a in enumerate(AMINO_ACIDS)}

### `one_hot_aa` (function)

Convert a sequence string into a (L, 20) one-hot tensor.

This is the simplest possible node feature. In later lessons we'll
swap this for ESM-2 embeddings (which capture much more biology).

In [ ]:
def one_hot_aa(sequence):
    n = len(sequence)
    x = torch.zeros(n, len(AMINO_ACIDS))
    for i, aa in enumerate(sequence):
        if aa in AA_TO_IDX:
            x[i, AA_TO_IDX[aa]] = 1.0
    return x

### `synthetic_helix_coords` (function)

Generate plausible alpha-helix Cα coordinates.

A real alpha-helix has rise ~1.5 Å per residue and ~100° rotation between
successive residues. This is just for demonstration — real proteins fold
into much more complex shapes. To use real coordinates, install biopython
and parse a PDB file (see "Things to experiment with" at the bottom).

In [ ]:
def synthetic_helix_coords(n_residues, rise=1.5, radius=2.3, turn_deg=100.0):
    angles = np.deg2rad(turn_deg * np.arange(n_residues))
    x = radius * np.cos(angles)
    y = radius * np.sin(angles)
    z = rise * np.arange(n_residues)
    return np.stack([x, y, z], axis=1)  # shape (n_residues, 3)

### `sequence_graph` (function)

Each residue is connected to its `window` nearest neighbours in the
sequence (in both directions, so the graph is undirected).

In [ ]:
def sequence_graph(sequence, window=2):
    n = len(sequence)
    edges = []
    for i in range(n):
        for d in range(1, window + 1):
            if i + d < n:
                # Add both directions — PyG expects directed edges; an
                # undirected edge is just two directed edges.
                edges.append((i, i + d))
                edges.append((i + d, i))
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=one_hot_aa(sequence), edge_index=edge_index)

### `contact_graph` (function)

Edge between residues whose Cα-Cα distance is below `threshold` Å.

8 Å is the standard "contact" cutoff in protein structure analysis.

In [ ]:
def contact_graph(sequence, coords, threshold=8.0):
    # Pairwise distance matrix (n, n).
    diffs = coords[:, None, :] - coords[None, :, :]
    dist = np.linalg.norm(diffs, axis=-1)

    # Exclude self-edges (i==j) and pairs above the threshold.
    mask = (dist < threshold) & (dist > 0)
    src, dst = np.where(mask)

    edge_index = torch.tensor(np.stack([src, dst]), dtype=torch.long)
    edge_attr = torch.tensor(dist[mask], dtype=torch.float32).unsqueeze(-1)

    return Data(
        x=one_hot_aa(sequence),
        edge_index=edge_index,
        edge_attr=edge_attr,
        pos=torch.tensor(coords, dtype=torch.float32),
    )

### `visualise_graph` (function)

Use networkx for layout + matplotlib for rendering.

In [ ]:
def visualise_graph(data, title, savepath):
    g = nx.Graph()
    for i in range(data.num_nodes):
        g.add_node(i)
    # data.edge_index is (2, E); each column is a directed edge. We add
    # them as undirected — networkx deduplicates automatically.
    for s, t in data.edge_index.t().tolist():
        g.add_edge(s, t)

    plt.figure(figsize=(7, 7))
    pos = nx.spring_layout(g, seed=42)
    nx.draw(
        g, pos,
        node_size=80, with_labels=False,
        node_color="skyblue", edge_color="lightgrey",
    )
    plt.title(title)
    plt.tight_layout()
    plt.savefig(savepath, dpi=100, bbox_inches="tight")
    plt.close()

### `main` (function)

In [ ]:
def main():
    os.makedirs("./results", exist_ok=True)

    sequence = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
    print(f"Sequence (length {len(sequence)}): {sequence}")

    # ---- 1. Sequence graph ----------------------------------------------
    seq_data = sequence_graph(sequence, window=2)
    print("\n[Sequence graph]")
    print(f"  num_nodes        = {seq_data.num_nodes}")
    print(f"  num_edges        = {seq_data.num_edges}  (counts each undirected edge twice)")
    print(f"  avg degree       = {seq_data.num_edges / seq_data.num_nodes:.1f}")
    print(f"  x.shape          = {tuple(seq_data.x.shape)}")
    print(f"  edge_index.shape = {tuple(seq_data.edge_index.shape)}")

    # ---- 2. Contact graph from synthetic 3D coordinates -----------------
    coords = synthetic_helix_coords(len(sequence))
    cnt_data = contact_graph(sequence, coords, threshold=8.0)
    print("\n[Contact graph from synthetic helix coords]")
    print(f"  num_nodes  = {cnt_data.num_nodes}")
    print(f"  num_edges  = {cnt_data.num_edges}")
    print(f"  avg degree = {cnt_data.num_edges / cnt_data.num_nodes:.1f}")
    print(f"  edge_attr  = mean {cnt_data.edge_attr.mean():.2f} Å, max {cnt_data.edge_attr.max():.2f} Å")

    # ---- 3. Visualise ---------------------------------------------------
    visualise_graph(seq_data, "Sequence graph (window=2)", "./results/gnn_l1_sequence_graph.png")
    visualise_graph(cnt_data, "Contact graph (threshold=8 Å, synthetic helix)", "./results/gnn_l1_contact_graph.png")
    print("\nSaved figures to ./results/gnn_l1_*.png")

    print(
        """
Things to experiment with:
- Build the contact graph from a REAL PDB structure (install biopython):
    from Bio.PDB import PDBList, PDBParser
    PDBList().retrieve_pdb_file("1ubq", file_format="pdb")  # ubiquitin, 76 residues
    # parse to get Cα coords + residue sequence, then call contact_graph().
- Vary the contact threshold (4 Å = direct contact; 8 Å = standard; 12 Å = loose).
- Vary the sequence-graph window — 1 = only-adjacent; 5 = wider context.
- Replace one_hot_aa() with ESM-2 embeddings (cf. plm_l1) — that's lesson 4's punch.
"""
    )

## Run the lesson

Execute everything above, then run `main()`.

In [ ]:
main()

## Bonus — real structures & an interactive explorer

Everything above used a *synthetic* helix. Let's swap in a **real** protein.
We download ubiquitin (PDB `1UBQ`, 76 residues) with **biopython**, parse its
Cα coordinates, and build the contact graph from genuine 3D geometry.

Then we add **ipywidgets** sliders so you can *see* how the graph changes as you
vary the contact threshold and the sequence-graph window — drag a slider and the
structure re-renders live.

> Needs `biopython` and `ipywidgets` (`pip install biopython ipywidgets`) and a
> network connection for the first download. If the download fails, the cell
> falls back to the synthetic helix so the explorer still works.

In [ ]:
# Download + parse a REAL structure (1UBQ) with biopython.
# Falls back to the synthetic helix from earlier if offline.
import os, urllib.request

os.makedirs('./results', exist_ok=True)
pdb_path = './results/1ubq.pdb'

try:
    from Bio.PDB import PDBParser
    from Bio.Data.IUPACData import protein_letters_3to1

    if not os.path.exists(pdb_path):
        urllib.request.urlretrieve('https://files.rcsb.org/download/1UBQ.pdb', pdb_path)

    three_to_one = {k.upper(): v for k, v in protein_letters_3to1.items()}
    parser = PDBParser(QUIET=True)
    chain = next(parser.get_structure('1ubq', pdb_path)[0].get_chains())

    seq_chars, coord_list = [], []
    for res in chain:
        # res.id[0] == ' ' keeps only standard residues (skips water/heteroatoms).
        name = res.get_resname().upper()
        if res.id[0] == ' ' and 'CA' in res and name in three_to_one:
            seq_chars.append(three_to_one[name])
            coord_list.append(res['CA'].get_coord())
    real_seq = ''.join(seq_chars)
    real_coords = np.asarray(coord_list, dtype=float)
    print(f'Parsed {len(real_seq)} residues from 1UBQ')
    print(real_seq)
except Exception as e:
    print(f'Could not load 1UBQ ({e!r}); falling back to the synthetic helix.')
    real_seq = 'MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG'
    real_coords = synthetic_helix_coords(len(real_seq))


In [ ]:
# Build the contact graph from the REAL coordinates, reusing contact_graph()
# defined earlier in this notebook.
real_data = contact_graph(real_seq, real_coords, threshold=8.0)
print(f'Real contact graph: {real_data.num_nodes} nodes, '
      f'{real_data.num_edges} directed edges, '
      f'avg degree {real_data.num_edges / real_data.num_nodes:.1f}')

# How many contacts are LONG-RANGE (|i-j| > 5)? These are exactly the ones a
# sequence-window graph can never capture — the whole point of using structure.
src, dst = real_data.edge_index
long_range = int(((dst - src).abs() > 5).sum().item()) // 2
print(f'Long-range contacts (|i-j| > 5): {long_range}')


### Interactive explorer

Drag the sliders. **Red** lines are *contacts* (Cα–Cα below the distance
threshold); **blue** lines are *sequence-window* edges (residues within `window`
of each other in the chain). The left panel is the 3D fold; the right panel is
the contact map. Watch the contact map fill in as you raise the threshold.

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers the 3d projection)

# Precompute the pairwise Cα-Cα distance matrix once; the sliders just re-threshold it.
_diffs = real_coords[:, None, :] - real_coords[None, :, :]
_dist = np.linalg.norm(_diffs, axis=-1)
_n = len(real_seq)


def explore(threshold=8.0, window=2, show_contacts=True, show_sequence=True):
    contact_mask = (_dist < threshold) & (_dist > 0)

    fig = plt.figure(figsize=(12, 5))

    # ---- left: 3D structure ----
    ax = fig.add_subplot(1, 2, 1, projection='3d')
    xs, ys, zs = real_coords[:, 0], real_coords[:, 1], real_coords[:, 2]
    ax.plot(xs, ys, zs, color='0.3', lw=1.0, alpha=0.6)              # backbone trace
    ax.scatter(xs, ys, zs, c=np.arange(_n), cmap='viridis', s=25)    # residues, N->C

    if show_contacts:
        ci, cj = np.where(np.triu(contact_mask, k=1))
        for i, j in zip(ci, cj):
            ax.plot(*zip(real_coords[i], real_coords[j]),
                    color='tab:red', alpha=0.25, lw=0.6)
    if show_sequence:
        for i in range(_n):
            for d in range(1, window + 1):
                if i + d < _n:
                    ax.plot(*zip(real_coords[i], real_coords[i + d]),
                            color='tab:blue', alpha=0.5, lw=0.8)
    ax.set_title(f'3D fold — contacts < {threshold:.1f} A')
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')

    # ---- right: contact map ----
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.imshow(contact_mask, cmap='Greys', origin='lower')
    ax2.set_title('Contact map')
    ax2.set_xlabel('residue j'); ax2.set_ylabel('residue i')
    plt.tight_layout(); plt.show()

    n_contacts = int(contact_mask.sum()) // 2
    n_seq = sum(_n - d for d in range(1, window + 1))
    print(f'threshold={threshold:.1f} A  ->  {n_contacts} contacts '
          f'(avg degree {contact_mask.sum() / _n:.1f})')
    print(f'window={window}        ->  {n_seq} sequence-window edges')


interact(
    explore,
    threshold=widgets.FloatSlider(min=3.0, max=15.0, step=0.5, value=8.0,
                                  description='threshold A', continuous_update=False),
    window=widgets.IntSlider(min=1, max=6, value=2, description='seq window'),
    show_contacts=widgets.Checkbox(value=True, description='show contacts'),
    show_sequence=widgets.Checkbox(value=True, description='show seq edges'),
);


### Rotatable 3D view with py3Dmol

The matplotlib explorer above is great for the contact-map overlay, but it's
awkward to rotate. **py3Dmol** (`pip install py3Dmol`) embeds a real WebGL
molecular viewer right in the notebook — **drag to rotate, scroll to zoom,
right-drag (or ctrl-drag) to pan.**

We draw the protein as a cartoon and overlay the **long-range contacts** as red
cylinders between Cα atoms. Move the threshold slider and the contacts redraw.

In [ ]:
# Build a PDB string for the viewer. If we downloaded a real PDB, use it (so the
# cartoon renders); otherwise synthesise a Ca-only PDB from the coordinates.
_one_to_three = {
    'A': 'ALA', 'R': 'ARG', 'N': 'ASN', 'D': 'ASP', 'C': 'CYS', 'Q': 'GLN',
    'E': 'GLU', 'G': 'GLY', 'H': 'HIS', 'I': 'ILE', 'L': 'LEU', 'K': 'LYS',
    'M': 'MET', 'F': 'PHE', 'P': 'PRO', 'S': 'SER', 'T': 'THR', 'W': 'TRP',
    'Y': 'TYR', 'V': 'VAL',
}


def _ca_only_pdb(seq, coords):
    """Minimal Ca-only PDB string (used only in the offline/synthetic fallback)."""
    lines = []
    for i, (aa, (x, y, z)) in enumerate(zip(seq, coords)):
        res3 = _one_to_three.get(aa, 'GLY')
        lines.append(
            'ATOM  %5d  CA  %3s A%4d    %8.3f%8.3f%8.3f  1.00  0.00           C'
            % (i + 1, res3, i + 1, x, y, z)
        )
    lines.append('END')
    return chr(10).join(lines)


if os.path.exists(pdb_path):
    with open(pdb_path) as fh:
        pdb_string = fh.read()
    has_cartoon = True
else:
    pdb_string = _ca_only_pdb(real_seq, real_coords)
    has_cartoon = False   # no backbone atoms -> cartoon can't be built
print(f'PDB string ready ({len(pdb_string.splitlines())} lines, cartoon={has_cartoon})')


In [ ]:
import py3Dmol

# Reuse the precomputed distance matrix (_dist, _n) from the matplotlib explorer.


def view_3dmol(threshold=8.0, min_seq_sep=5, style='cartoon', show_contacts=True):
    """Render an interactive 3Dmol viewer: cartoon + long-range contact cylinders.

    Drag to rotate, scroll to zoom, right-drag to pan.
    min_seq_sep hides trivial local contacts (|i - j| <= min_seq_sep).
    """
    view = py3Dmol.view(width=560, height=460)
    view.addModel(pdb_string, 'pdb')

    # Base representation of the protein.
    if style == 'cartoon' and has_cartoon:
        view.setStyle({'cartoon': {'color': 'spectrum'}})
    elif style == 'sphere':
        view.setStyle({'sphere': {'radius': 0.8, 'colorscheme': 'Jmol'}})
    else:  # 'stick', or cartoon fallback when there are no backbone atoms
        view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'radius': 0.4}})

    # Overlay long-range contacts as red cylinders between Ca atoms.
    n_drawn = 0
    if show_contacts:
        mask = (_dist < threshold) & (_dist > 0)
        ci, cj = np.where(np.triu(mask, k=1))
        for i, j in zip(ci, cj):
            if abs(int(i) - int(j)) <= min_seq_sep:
                continue
            a, b = real_coords[i], real_coords[j]
            view.addCylinder({
                'start': {'x': float(a[0]), 'y': float(a[1]), 'z': float(a[2])},
                'end':   {'x': float(b[0]), 'y': float(b[1]), 'z': float(b[2])},
                'radius': 0.12, 'color': 'red', 'opacity': 0.7,
            })
            n_drawn += 1

    view.zoomTo()
    view.show()
    print(f'threshold={threshold:.1f} A, min_seq_sep={min_seq_sep}  ->  '
          f'{n_drawn} long-range contact cylinders drawn')


interact(
    view_3dmol,
    threshold=widgets.FloatSlider(min=3.0, max=15.0, step=0.5, value=8.0,
                                  description='threshold A', continuous_update=False),
    min_seq_sep=widgets.IntSlider(min=0, max=12, value=5, description='min |i-j|',
                                  continuous_update=False),
    style=widgets.Dropdown(options=['cartoon', 'stick', 'sphere'], value='cartoon',
                           description='style'),
    show_contacts=widgets.Checkbox(value=True, description='show contacts'),
);
